# Lesson 9 — Fine-tune the assembled ACT model

You will update the assembled ACT model from Lesson 8 on the shared `bowl_plate` task. You will keep a fixed validation split, save a baseline and an adapted checkpoint, load the saved processors, and compare validation error and a LIBERO rollout.

This lesson assumes strong Python skills and basic machine learning. It explains the PyTorch calls used in the exercise. The real run uses the frozen local dataset. It does not download data.

## Model and training path

ACT maps the current state and two camera images to a future action chunk. For batch size **B**, chunk length **T = 20**, and action width **D = 7**, the predicted actions have shape `(B, T, D)`. `action_is_pad` has shape `(B, T)`; `True` marks action targets that do not exist because the episode ended. The state has shape `(B, 8)`. Each image has shape `(B, 3, H, W)` after the saved processor. `H` and `W` are pixels.

Fine-tuning starts with parameter tensors θ from a saved ACT checkpoint. At update *k*, the optimizer applies `θₖ₊₁ = θₖ − η · gₖ`, where *η* is the learning rate in optimizer units and *gₖ* is the gradient of the loss with respect to θ. The loss is masked action L1 plus a weighted latent KL term:

`L = Σ(b,t,d) v_bt |â_btd − a_btd| / (D · Σ(b,t) v_bt) + β · KL(q(z|s,a) || N(0,I))`.

Here `a` is the normalized target action, `â` is the predicted action, `v_bt = 1 − m_bt` is 1 for a real action and 0 for padding, `m_bt` is the padding mask, `β` is the KL weight, and `z` is the latent vector. The mask applies to the action term. It prevents padded targets from changing either the numerator or denominator. The KL term is averaged over the batch and summed over latent dimensions by the supplied loss function.

The inputs and targets already pass through the saved ACT preprocessor. Image pixels are scaled from integer range 0–255 to float range 0–1; this is image scaling. State and action values use means and standard deviations saved from training episodes; this is feature standardization. These steps occur before model prediction and loss calculation. The postprocessor converts predicted normalized actions back to controller units. The environment then clips values to its valid command range. The validation split does not fit statistics and does not update parameters.

A small example: with two valid actions and one padded action, action errors `0.2`, `0.4`, and `9.0` on one dimension give masked L1 `(0.2 + 0.4) / 2 = 0.3`; the padded error `9.0` has no effect. The action tensor still has all three time positions. Its mask decides which positions count.

```text
LIBERO rows -> saved preprocessor -> ACTModel -> masked loss -> optimizer update
                   |                               |
                   +-- saved statistics            +-- train split only
fixed validation rows -> same preprocessor -> error before/after/reload
saved model + saved processors -> LIBERO rollout before and after
```

The training data uses 10 frames per second as its storage label. LIBERO control runs at 20 steps per second. Each relative action is one environment step. The dataset storage rate does not set an action repeat rule. We do not decimate actions here. A 200-step rollout therefore has at most 10 seconds of simulated control time, apart from reset time.

## Exercise: implement a bounded update path

Implement `fine_tune_updates`. Each update must transfer only the current batch to the selected device, call the ACT loss, backpropagate, clip the gradient norm, and update the model. `steps` is an update count. `max_seconds` is a wall-clock budget checked by the supplied training loop before each update. The function returns metrics for updates that completed before that limit. The fixed CPU fixture below checks parameter change and reduced action error. The required experiment uses this same function with the assembled ACT model and real data.

In [ ]:
import json
import os
import sys
import time
from collections import deque
from collections.abc import Mapping, Sequence
from dataclasses import asdict
from pathlib import Path
from typing import cast
from uuid import uuid4

import numpy as np
import torch
from torch import Tensor, nn


def find_workspace(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "manifests" / "act_references.yaml").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the VLA course workspace.")


WORKSPACE = find_workspace()
sys.path.insert(0, str(WORKSPACE / "src"))
HF_ROOT = WORKSPACE / ".local-data" / "hf"
os.environ["HF_HUB_CACHE"] = str(HF_ROOT / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_ROOT / "datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("Workspace:", WORKSPACE)
print("PyTorch:", torch.__version__)

In [ ]:
from lerobot.policies.act import ACTConfig, ACTPolicy
from lerobot.processor import PolicyProcessorPipeline, load_pretrained_policy_processors

from vla_lab.evaluation.demo import encode_rgb_video
from vla_lab.policy.act import make_act_processors
from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.act_model import ACTModel, act_from_lerobot
from vla_lab.teaching.interfaces import ACTPredictionModel, LossCallback, ModelT
from vla_lab.teaching.live import load_act_batches
from vla_lab.teaching.training import act_loss_fn, masked_action_l1, train_loop

## Exercise check

This small check uses fixed CPU tensors. It does not need model weights, a simulator, or local episodes.

In [ ]:
def fine_tune_updates(
    model: ModelT,
    batches: Sequence[Mapping[str, Tensor]],
    optimizer: torch.optim.Optimizer,
    *,
    steps: int,
    device: torch.device,
    kl_weight: float,
    max_grad_norm: float,
    max_seconds: float | None = None,
) -> list[dict[str, float]]:
    """Run bounded ACT updates and move each batch only when it is used."""
    if not batches:
        raise ValueError("batches must contain at least one training batch")
    if steps < 0 or kl_weight < 0 or max_grad_norm <= 0:
        raise ValueError("steps, KL weight, or gradient limit is invalid")

    def device_loss(
        candidate: ModelT,
        cpu_batch: Mapping[str, Tensor],
        *,
        noise: Tensor | None = None,
        time: Tensor | None = None,
    ) -> tuple[Tensor, Mapping[str, Tensor | float]]:
        batch = {name: value.to(device) for name, value in cpu_batch.items()}
        loss, metrics = act_loss_fn(
            cast(ACTPredictionModel, candidate),
            batch,
            noise=noise,
            time=time,
            kl_weight=kl_weight,
        )
        return loss, dict(metrics)

    return train_loop(
        model,
        batches,
        optimizer,
        cast(LossCallback[ModelT], device_loss),
        steps=steps,
        max_grad_norm=max_grad_norm,
        max_seconds=max_seconds,
    )


class MiniACT(nn.Module):
    """A small predictor for the update check; it has the same output contract."""

    def __init__(self) -> None:
        super().__init__()
        self.action_bias = nn.Parameter(torch.zeros(2))

    def forward(self, batch: Mapping[str, Tensor]) -> tuple[Tensor, tuple[Tensor, Tensor]]:
        target = batch["action"]
        prediction = self.action_bias.view(1, 1, 2).expand_as(target)
        zero = torch.zeros((target.shape[0], 1), device=target.device)
        return prediction, (zero, zero)


mini_batch = {
    "action": torch.tensor(
        [[[0.4, -0.2], [0.4, -0.2], [0.4, -0.2]], [[0.4, -0.2], [0.4, -0.2], [0.4, -0.2]]]
    ),
    "action_is_pad": torch.tensor([[False, False, False], [False, False, True]]),
}
mini_model = MiniACT()
mini_optimizer = torch.optim.AdamW(mini_model.parameters(), lr=0.05)
with torch.inference_mode():
    before = masked_action_l1(
        mini_model(mini_batch)[0], mini_batch["action"], mini_batch["action_is_pad"]
    )
mini_history = fine_tune_updates(
    mini_model,
    [mini_batch],
    mini_optimizer,
    steps=8,
    device=torch.device("cpu"),
    kl_weight=0.01,
    max_grad_norm=1.0,
)
with torch.inference_mode():
    after = masked_action_l1(
        mini_model(mini_batch)[0], mini_batch["action"], mini_batch["action_is_pad"]
    )
assert len(mini_history) == 8, "Return one metric row for each completed update."
assert after < before, "Repeated updates must reduce error on the fixed valid targets."
print(
    {
        "updates": len(mini_history),
        "valid_action_l1_before": float(before),
        "valid_action_l1_after": float(after),
    }
)

## Full experiment — prepare a baseline

The real run uses 256 training batches and 64 fixed validation batches. It starts from the prepared `bowl_plate` ACT checkpoint and copies its weights into the assembled `ACTModel` from Lesson 8. It saves the model state, model settings, optimizer state, frozen data identity, and both processor pipelines before the first update. The baseline validation and rollout use the processors loaded back from that saved checkpoint.

In [ ]:
TASK_ALIAS = "bowl_plate"
SOURCE_CHECKPOINT = WORKSPACE / ".local-data/runs/act-bowl-plate-probe-200/checkpoint-000200"
CHUNK_SIZE = 20
BATCH_SIZE = 8
TRAIN_BATCH_COUNT = 256
VALIDATION_BATCH_COUNT = 64
UPDATE_BUDGET = 1500
UPDATE_BLOCK = 100
WALL_LIMIT_S = 2 * 60 * 60
SAVE_RESERVE_S = 180
SEED = 29
KL_WEIGHT = 0.01
LEARNING_RATE = 1e-5
MAX_GRAD_NORM = 1.0
RUN_STARTED = time.monotonic()

if not torch.cuda.is_available():
    raise RuntimeError("The required full adaptation experiment needs CUDA.")
GPU_BYTES = torch.cuda.get_device_properties(0).total_memory
if GPU_BYTES > 48 * 1024**3:
    torch.cuda.set_per_process_memory_fraction((48 * 1024**3) / GPU_BYTES)
DEVICE = torch.device("cuda")
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.cuda.reset_peak_memory_stats(DEVICE)
OUTPUT = WORKSPACE / ".local-data/course-runs/unit09" / uuid4().hex
OUTPUT.mkdir(parents=True, exist_ok=False)

DATA = load_act_batches(
    TASK_ALIAS,
    WORKSPACE,
    batch_size=BATCH_SIZE,
    chunk_size=CHUNK_SIZE,
    train_batches=TRAIN_BATCH_COUNT,
    validation_batches=VALIDATION_BATCH_COUNT,
    seed=SEED,
)
print("Frozen train/validation identity:", DATA.metadata)
STARTER_POLICY = ACTPolicy.from_pretrained(SOURCE_CHECKPOINT, local_files_only=True)
STARTER_POLICY.config = cast(ACTConfig, STARTER_POLICY.config)
STARTER_POLICY.config = cast(ACTConfig, STARTER_POLICY.config)
PREPROCESSOR, POSTPROCESSOR = make_act_processors(
    STARTER_POLICY.config, TASK_ALIAS, WORKSPACE / "configs"
)
MODEL = act_from_lerobot(STARTER_POLICY.model).to(DEVICE)
OPTIMIZER = torch.optim.AdamW(MODEL.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)


def validation_action_l1(
    model: ACTModel,
    batches: Sequence[Mapping[str, Tensor]],
    device: torch.device,
) -> float:
    model.eval()
    values = []
    with torch.inference_mode():
        for cpu_batch in batches:
            batch = {name: value.to(device) for name, value in cpu_batch.items()}
            prediction, _latent = model(batch)
            values.append(masked_action_l1(prediction, batch["action"], batch["action_is_pad"]))
    return float(torch.stack(values).mean().cpu())


class AssembledACTPolicy:
    """Send the predicted chunk one action at a time to the episode runner."""

    def __init__(self, model: ACTModel) -> None:
        self.model = model
        self.actions: deque[Tensor] = deque()

    def reset(self) -> None:
        self.actions.clear()

    def select_action(self, batch: dict[str, Tensor]) -> Tensor:
        if not self.actions:
            chunk, _latent = self.model(batch)
            self.actions.extend(chunk[0, index] for index in range(CHUNK_SIZE))
        return self.actions.popleft().unsqueeze(0)


def save_assembled_checkpoint(
    model: ACTModel,
    optimizer: torch.optim.Optimizer,
    preprocessor: PolicyProcessorPipeline,
    postprocessor: PolicyProcessorPipeline,
    destination: Path,
    *,
    updates: int,
    status: str,
) -> Path:
    destination.mkdir(parents=True, exist_ok=False)
    torch.save(
        {"model": model.state_dict(), "optimizer": optimizer.state_dict()}, destination / "state.pt"
    )
    preprocessor.save_pretrained(destination)
    postprocessor.save_pretrained(destination)
    manifest = {
        "schema_version": 1,
        "model": "assembled-act",
        "task_alias": TASK_ALIAS,
        "source_checkpoint": str(SOURCE_CHECKPOINT),
        "updates": updates,
        "status": status,
        "act_settings": asdict(model.settings),
        "seed": SEED,
        "learning_rate": LEARNING_RATE,
        "weight_decay": 1e-4,
        "kl_weight": KL_WEIGHT,
        "max_grad_norm": MAX_GRAD_NORM,
        "optimizer": "AdamW",
        "dataset_identity": DATA.metadata,
    }
    (destination / "manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n"
    )
    return destination


def load_assembled_checkpoint(
    checkpoint: Path,
    device: torch.device,
) -> tuple[ACTModel, PolicyProcessorPipeline, PolicyProcessorPipeline, dict[str, object]]:
    manifest: dict[str, object] = json.loads((checkpoint / "manifest.json").read_text())
    source = ACTPolicy.from_pretrained(SOURCE_CHECKPOINT, local_files_only=True)
    model = act_from_lerobot(source.model)
    payload = torch.load(checkpoint / "state.pt", map_location=device, weights_only=True)
    model.load_state_dict(payload["model"], strict=True)
    model = model.to(device).eval()
    preprocessor, postprocessor = load_pretrained_policy_processors(checkpoint)
    return model, preprocessor, postprocessor, manifest


def run_assembled_rollout(
    model: ACTModel,
    preprocessor: PolicyProcessorPipeline,
    postprocessor: PolicyProcessorPipeline,
    output_dir: Path,
) -> dict[str, object]:
    output_dir.mkdir(parents=True, exist_ok=False)
    environment = open_libero_task(
        TASK_ALIAS,
        0,
        WORKSPACE / "configs",
        WORKSPACE / ".local-data",
        control_hz=20,
    )
    frames: list[np.ndarray] = []
    try:
        result = run_policy_episode(
            environment,
            AssembledACTPolicy(model.eval()),
            preprocessor,
            postprocessor,
            TASK_ALIAS,
            0,
            WORKSPACE / "configs",
            partition="development",
            max_steps=200,
            on_frame=frames.append,
            reset_seed=20261006,
            control_hz=20,
        )
    finally:
        environment.close()
    video = encode_rgb_video(frames, output_dir / "rollout.mp4", fps=20)
    return {
        "success": result.success,
        "steps": result.steps,
        "termination_reason": result.termination_reason,
        "control_hz": result.control_hz,
        "simulated_duration_s": result.simulated_duration_s,
        "video": str(video),
    }


BASELINE_DIR = save_assembled_checkpoint(
    MODEL,
    OPTIMIZER,
    PREPROCESSOR,
    POSTPROCESSOR,
    OUTPUT / "baseline",
    updates=0,
    status="baseline",
)
BASELINE_MODEL, BASELINE_PREPROCESSOR, BASELINE_POSTPROCESSOR, BASELINE_MANIFEST = (
    load_assembled_checkpoint(
        BASELINE_DIR,
        DEVICE,
    )
)
BEFORE_VALIDATION_L1 = validation_action_l1(BASELINE_MODEL, DATA.validation, DEVICE)
ROLLOUT_BEFORE = run_assembled_rollout(
    BASELINE_MODEL,
    BASELINE_PREPROCESSOR,
    BASELINE_POSTPROCESSOR,
    OUTPUT / "rollout-before",
)
print("Baseline:", {"validation_action_l1": BEFORE_VALIDATION_L1, "rollout": ROLLOUT_BEFORE})

## Full experiment — fine-tune on training episodes

Each block uses the learner function from the exercise. The validation batches stay outside the optimizer. The deadline includes setup time and keeps three minutes for checkpoint save and evaluation. The training loop checks its time budget before each update. It may finish the update that began just before the deadline. Each completed block writes a checkpoint with its optimizer state and update count.

In [ ]:
DEADLINE = RUN_STARTED + WALL_LIMIT_S - SAVE_RESERVE_S
updates = 0
history: list[dict[str, float]] = []
LAST_CHECKPOINT: Path | None = None
while updates < UPDATE_BUDGET and time.monotonic() < DEADLINE:
    remaining_s = DEADLINE - time.monotonic()
    if remaining_s <= 0:
        break
    requested = min(UPDATE_BLOCK, UPDATE_BUDGET - updates)
    # Continue through all batches when a checkpoint block ends.
    batch_offset = updates % len(DATA.train)
    block_batches = DATA.train[batch_offset:] + DATA.train[:batch_offset]
    block_history = fine_tune_updates(
        MODEL,
        block_batches,
        OPTIMIZER,
        steps=requested,
        device=DEVICE,
        kl_weight=KL_WEIGHT,
        max_grad_norm=MAX_GRAD_NORM,
        max_seconds=remaining_s,
    )
    history.extend(block_history)
    updates += len(block_history)
    status = "complete" if updates == UPDATE_BUDGET else "incomplete"
    LAST_CHECKPOINT = save_assembled_checkpoint(
        MODEL,
        OPTIMIZER,
        PREPROCESSOR,
        POSTPROCESSOR,
        OUTPUT / f"checkpoint-update-{updates:05d}",
        updates=updates,
        status=status,
    )
    if block_history:
        print(
            f"updates={updates}/{UPDATE_BUDGET} elapsed_s={time.monotonic() - RUN_STARTED:.1f} loss={block_history[-1]['loss']:.5f}"
        )
    if len(block_history) < requested:
        break

TRAINING_COMPLETE = updates == UPDATE_BUDGET
if LAST_CHECKPOINT is None:
    LAST_CHECKPOINT = save_assembled_checkpoint(
        MODEL,
        OPTIMIZER,
        PREPROCESSOR,
        POSTPROCESSOR,
        OUTPUT / "checkpoint-update-00000",
        updates=0,
        status="incomplete",
    )
print(
    {
        "training_complete": TRAINING_COMPLETE,
        "updates": updates,
        "update_budget": UPDATE_BUDGET,
        "checkpoint": str(LAST_CHECKPOINT),
    }
)

## Full experiment — reload and compare

Load the last saved model and both saved processor pipelines. Compare it with the baseline on the same held-out validation batches and the same development state. The record keeps validation action error separate from LIBERO success. A partial run stays marked incomplete.

In [ ]:
if LAST_CHECKPOINT is None:
    raise RuntimeError("The training stage did not save a checkpoint.")
FINAL_CHECKPOINT: Path = LAST_CHECKPOINT
FINAL_MODEL, FINAL_PREPROCESSOR, FINAL_POSTPROCESSOR, FINAL_MANIFEST = load_assembled_checkpoint(
    FINAL_CHECKPOINT,
    DEVICE,
)
AFTER_VALIDATION_L1 = validation_action_l1(FINAL_MODEL, DATA.validation, DEVICE)
ROLLOUT_AFTER = run_assembled_rollout(
    FINAL_MODEL,
    FINAL_PREPROCESSOR,
    FINAL_POSTPROCESSOR,
    OUTPUT / "rollout-after",
)
record = {
    "status": "complete" if TRAINING_COMPLETE else "incomplete",
    "task_alias": TASK_ALIAS,
    "updates": updates,
    "update_budget": UPDATE_BUDGET,
    "elapsed_s": time.monotonic() - RUN_STARTED,
    "device": str(DEVICE),
    "gpu_peak_memory_gb": torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
    "dataset_identity": DATA.metadata,
    "validation_action_l1_before": BEFORE_VALIDATION_L1,
    "validation_action_l1_after": AFTER_VALIDATION_L1,
    "validation_action_l1_change": AFTER_VALIDATION_L1 - BEFORE_VALIDATION_L1,
    "baseline_manifest": BASELINE_MANIFEST,
    "final_manifest": FINAL_MANIFEST,
    "rollout_before": ROLLOUT_BEFORE,
    "rollout_after": ROLLOUT_AFTER,
    "checkpoint": str(LAST_CHECKPOINT),
}
(OUTPUT / "adaptation_record.json").write_text(json.dumps(record, indent=2, sort_keys=True) + "\n")
print(json.dumps(record, indent=2, sort_keys=True))

## Limits

A lower validation error does not prove task success. One development rollout does not prove performance on all states. The run reports the checkpoint, processors, episode split, validation error, rollout result, simulated control time, elapsed wall time, and peak GPU memory.